In [1]:
import os
os.environ['HF_HOME'] = '/projectnb/vkolagrp/skowshik/.cache/'
import sys
import logging
import argparse
import warnings
warnings.filterwarnings("ignore")
import yaml
import torch
import json
import pandas as pd
import torch.distributed as dist

from torch.nn.parallel import DistributedDataParallel as DDP
from datetime import datetime, timedelta
from huggingface_hub import login
from lib.model_loader import load_model, trainer_loader
from lib.data_loader import data_loader_from_json
from utils.utils import CustomStream, load_config
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from lib.model_loader import load_model, load_model_eval, trainer_loader
from utils.utils import load_json

In [3]:
config = load_config(file_name="../../code/training/config/config.yml")
new_model = '../../ckpt/fine_tuned_v3/checkpoint-80000/'
model, tokenizer = load_model_eval(config, new_model)
sysmsg = config.get("sysmsg")
usermsg_prefix = config.get("usermsg_prefix")
dataset_path = "../../data/nacc_unique_with_llama_summaries_2_only_np.json"
data = load_json(dataset_path)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.89s/it]


In [13]:
# index = 0 #, 0 # LBD DE
# index = 3 # MCI
# index = 12 # AD DE
index = 1 # NC
# index = 242

# messages = [
#     # {"role": "system", "content": sysmsg},
#     {"role": "user", "content": f"{usermsg_prefix} {data[index]['patient_LLAMA_SUMMARY']}"},
# ]

messages = [
    # {"role": "system", "content": sysmsg},
    {"role": "user", "content": f"Answer to the point. Does this patient have depression?\n{data[index]['patient_LLAMA_SUMMARY']}."},
]
# messages = [
#     # {"role": "system", "content": "You are a math expert"},
#     {"role": "user", "content": "which is greater, 3.11 vs. 3.8?"}
# ]

In [14]:
prompt = tokenizer.apply_chat_template(
    messages, 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=config.get("max_new_tokens"),
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

**Conclusion:**

Based on the provided information, the patient's cognitive status is classified as "normal." The patient's Mini-Mental State Examination (MMSE) score is 25, which is within the normal range. The patient's performance on the Boston Naming Test, Logical Memory IIA - Delayed, and Digit Span tests is also within normal limits. The patient's Trail Making Test Part A and Part B scores are physical problems, but this is not a cognitive impairment. The patient's WAIS-R Digit Symbol score is also physical problem, but this is not a cognitive impairment.

The patient's Geriatric Depression Scale (GDS) score is 1, indicating no depression. The patient's Functional Assessment Scale (FAS) score is also normal, indicating no difficulty with daily activities.

The patient's genetic testing reveals an APOE genotype of (e3 e3), which is not associated with an increased risk of Alzheimer's disease.

The patient's physical and neurological exam findings are normal, with no abnormal findi